# Preparación de CSV para Tableau  
Este notebook limpia y transforma los datasets de **FGJ** y **pobreza multidimensional/MMIP** para generar archivos usables en Tableau.

## Archivos esperados
- `datasets/carpetasFGJ_acumulado_2025_01.csv`
- `datasets/enigh_16_20.csv`

## Archivos de salida
Se crearán dentro de `csv_tableau/`:
1. `fgj_tableau_mensual_alcaldia.csv`
2. `pobreza_tableau_alcaldia_anio.csv`
3. `panel_tableau_alcaldia_anio.csv` (integrado, opcional pero muy útil)

## Qué hace el notebook
- Limpia años y alcaldías de FGJ.
- Crea categorías visuales de robo patrimonial y subtipo de robo.
- Agrega FGJ a nivel **alcaldía-año-mes-categoría** para hacerlo ligero y útil en Tableau.
- Filtra la base social a CDMX.
- Calcula indicadores sociales agregados por **alcaldía-año** usando el `factor`.
- Integra ambas fuentes en un panel único para análisis conjunto.


In [1]:
from pathlib import Path
import json
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd

# -----------------------------
# Rutas
# -----------------------------
if (Path.cwd() / "datasets").exists():
    BASE_DIR = Path.cwd()
elif (Path.cwd().parent / "datasets").exists():
    BASE_DIR = Path.cwd().parent
else:
    raise FileNotFoundError("No se encontró la carpeta 'datasets' desde el directorio actual ni desde su padre.")

DATA_DIR = BASE_DIR / "datasets"
OUT_DIR = BASE_DIR / "csv_tableau"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FGJ_FILE = DATA_DIR / "carpetasFGJ_acumulado_2025_01.csv"
POB_FILE = DATA_DIR / "enigh_16_20.csv"

print("BASE_DIR:", BASE_DIR)
print("FGJ_FILE:", FGJ_FILE, "| existe:", FGJ_FILE.exists())
print("POB_FILE:", POB_FILE, "| existe:", POB_FILE.exists())
print("OUT_DIR:", OUT_DIR)


BASE_DIR: d:\MineriaDatos\DMProject
FGJ_FILE: d:\MineriaDatos\DMProject\datasets\carpetasFGJ_acumulado_2025_01.csv | existe: True
POB_FILE: d:\MineriaDatos\DMProject\datasets\enigh_16_20.csv | existe: True
OUT_DIR: d:\MineriaDatos\DMProject\csv_tableau


In [2]:
# -----------------------------
# Utilidades
# -----------------------------
ENCODINGS = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

def read_csv_flexible(path, **kwargs):
    last_error = None
    for enc in ENCODINGS:
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            print(f"[OK] Leído con encoding: {enc}")
            return df, enc
        except Exception as e:
            last_error = e
    raise last_error

def normalize_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip().upper()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    text = " ".join(text.split())
    return text if text else None

MESES_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

ALCALDIA_MAP = {
    "ALVARO OBREGON": "Álvaro Obregón",
    "AZCAPOTZALCO": "Azcapotzalco",
    "BENITO JUAREZ": "Benito Juárez",
    "COYOACAN": "Coyoacán",
    "CUAJIMALPA DE MORELOS": "Cuajimalpa de Morelos",
    "CUAUHTEMOC": "Cuauhtémoc",
    "GUSTAVO A MADERO": "Gustavo A. Madero",
    "IZTACALCO": "Iztacalco",
    "IZTAPALAPA": "Iztapalapa",
    "LA MAGDALENA CONTRERAS": "La Magdalena Contreras",
    "MAGDALENA CONTRERAS": "La Magdalena Contreras",
    "MIGUEL HIDALGO": "Miguel Hidalgo",
    "MILPA ALTA": "Milpa Alta",
    "TLAHUAC": "Tláhuac",
    "TLALPAN": "Tlalpan",
    "VENUSTIANO CARRANZA": "Venustiano Carranza",
    "XOCHIMILCO": "Xochimilco",
}

MUNICIPIO_CDMX_MAP = {
    2: "Azcapotzalco",
    3: "Coyoacán",
    4: "Cuajimalpa de Morelos",
    5: "Gustavo A. Madero",
    6: "Iztacalco",
    7: "Iztapalapa",
    8: "La Magdalena Contreras",
    9: "Milpa Alta",
    10: "Álvaro Obregón",
    11: "Tláhuac",
    12: "Tlalpan",
    13: "Xochimilco",
    14: "Benito Juárez",
    15: "Cuauhtémoc",
    16: "Miguel Hidalgo",
    17: "Venustiano Carranza",
}

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(obj), f, ensure_ascii=False, indent=2)

def weighted_mean(series, weights):
    s = pd.to_numeric(series, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce")
    mask = s.notna() & w.notna()
    if mask.sum() == 0 or w[mask].sum() == 0:
        return np.nan
    return float(np.average(s[mask], weights=w[mask]))

def pct_null(df, cols):
    rows = []
    for c in cols:
        if c in df.columns:
            rows.append({
                "columna": c,
                "nulos": int(df[c].isna().sum()),
                "porcentaje_nulos": round(df[c].isna().mean() * 100, 4)
            })
    return pd.DataFrame(rows).sort_values("porcentaje_nulos", ascending=False)


## 1. Limpieza de FGJ y construcción del CSV para Tableau

### Diseño del archivo de salida
El archivo `fgj_tableau_mensual_alcaldia.csv` se exportará ya **agregado** para que Tableau lo mueva con facilidad.

Cada fila representará una combinación de:
- alcaldía,
- año,
- mes,
- categoría general,
- subtipo de robo.

### Variables visuales incluidas
- `categoria_general`: separa **Robo patrimonial** de **Otro delito**.
- `subtipo_robo`: agrupa robos para análisis visual.
- `conteo`: número de registros.
- `conteo_robos`: útil para filtrar directamente el fenómeno principal.
- `share_robos_mes_alcaldia`: porcentaje de robos sobre el total delictivo de ese mes y alcaldía.

Esto permite hacer dashboards mucho más claros que usar la base cruda.


In [3]:
# -----------------------------
# Leer FGJ
# -----------------------------
fgj, fgj_encoding = read_csv_flexible(FGJ_FILE, low_memory=False)
print(fgj.shape)
fgj.head(2)


[OK] Leído con encoding: utf-8
(2098743, 21)


,anio_inicio,mes_inicio,fecha_inicio,hora_inicio,anio_hecho,mes_hecho,fecha_hecho,hora_hecho,delito,categoria_delito,...,fiscalia,agencia,unidad_investigacion,colonia_hecho,colonia_catalogo,alcaldia_hecho,alcaldia_catalogo,municipio_hecho,latitud,longitud
0,2016,Enero,2016-01-01,00:00:00,2015.0,Diciembre,2015-12-31,16:30:00,LESIONES CULPOSAS POR TRANSITO VEHICULAR EN CO...,DELITO DE BAJO IMPACTO,...,INVESTIGACIÓN EN TLALPAN,TLP-4,UI-2CD,JARDINES EN LA MONTAÑA,Jardines En La Montaña,TLALPAN,NaN,CDMX,19.30086,-99.20877
1,2016,Enero,2016-01-01,00:00:00,2015.0,Diciembre,2015-12-31,22:40:00,ROBO A PASAJERO A BORDO DE TAXI CON VIOLENCIA,ROBO A PASAJERO A BORDO DE TAXI CON VIOLENCIA,...,INVESTIGACIÓN EN TLALPAN,TLP-1,UI-2CD,LOMAS DE PADIERNA,Lomas De Padierna,TLALPAN,NaN,CDMX,19.29003,-99.21748


In [4]:
# -----------------------------
# Resolver nombres de columnas de FGJ
# -----------------------------
cols = {normalize_text(c): c for c in fgj.columns}

def get_col(*aliases):
    for a in aliases:
        key = normalize_text(a)
        if key in cols:
            return cols[key]
    return None

c_delito = get_col("delito")
c_fecha_hecho = get_col("fecha_hecho")
c_anio_hecho = get_col("anio_hecho", "año_hecho")
c_mes_hecho = get_col("mes_hecho")
c_anio_inicio = get_col("anio_inicio", "año_inicio")
c_alcaldia_hecho = get_col("alcaldia_hech", "alcaldia_hecho")
c_alcaldia_cat = get_col("alcaldia_cata", "alcaldia_cat")
c_latitud = get_col("latitud")
c_longitud = get_col("longitud")

resueltas_fgj = {
    "delito": c_delito,
    "fecha_hecho": c_fecha_hecho,
    "anio_hecho": c_anio_hecho,
    "mes_hecho": c_mes_hecho,
    "anio_inicio": c_anio_inicio,
    "alcaldia_hecho": c_alcaldia_hecho,
    "alcaldia_cat": c_alcaldia_cat,
    "latitud": c_latitud,
    "longitud": c_longitud,
}
resueltas_fgj


{'delito': 'delito',
 'fecha_hecho': 'fecha_hecho',
 'anio_hecho': 'anio_hecho',
 'mes_hecho': 'mes_hecho',
 'anio_inicio': 'anio_inicio',
 'alcaldia_hecho': 'alcaldia_hecho',
 'alcaldia_cat': None,
 'latitud': 'latitud',
 'longitud': 'longitud'}

In [5]:
# -----------------------------
# Normalización temporal y territorial de FGJ
# -----------------------------
fgj["delito_norm"] = fgj[c_delito].map(normalize_text)

if c_fecha_hecho:
    fgj["fecha_hecho_clean"] = pd.to_datetime(fgj[c_fecha_hecho], errors="coerce", dayfirst=True)
else:
    fgj["fecha_hecho_clean"] = pd.NaT

if c_anio_hecho:
    fgj["anio_hecho_num"] = pd.to_numeric(fgj[c_anio_hecho], errors="coerce")
else:
    fgj["anio_hecho_num"] = np.nan

# Año limpio: prioriza anio_hecho si está en rango válido; si no, usa fecha_hecho
fgj["anio_hecho_clean"] = np.where(
    fgj["anio_hecho_num"].between(2016, 2025, inclusive="both"),
    fgj["anio_hecho_num"],
    fgj["fecha_hecho_clean"].dt.year
)
fgj["anio_hecho_clean"] = pd.to_numeric(fgj["anio_hecho_clean"], errors="coerce")

# Mes limpio
fgj["mes_num"] = fgj["fecha_hecho_clean"].dt.month
if c_mes_hecho and fgj["mes_num"].isna().mean() > 0.05:
    # si faltan muchos meses por fecha, intenta mapear desde texto
    mes_txt = fgj[c_mes_hecho].astype(str).str.strip().str.upper()
    inv_meses = {v.upper(): k for k, v in MESES_ES.items()}
    fgj["mes_num_alt"] = mes_txt.map(inv_meses)
    fgj["mes_num"] = fgj["mes_num"].fillna(fgj["mes_num_alt"])
fgj["mes_num"] = pd.to_numeric(fgj["mes_num"], errors="coerce")
fgj["mes_nombre"] = fgj["mes_num"].map(MESES_ES)

# Alcaldía final
if c_alcaldia_hecho:
    fgj["alcaldia_hecho_norm_raw"] = fgj[c_alcaldia_hecho].map(normalize_text)
else:
    fgj["alcaldia_hecho_norm_raw"] = None

if c_alcaldia_cat:
    fgj["alcaldia_cat_norm_raw"] = fgj[c_alcaldia_cat].map(normalize_text)
else:
    fgj["alcaldia_cat_norm_raw"] = None

fgj["alcaldia_norm_raw"] = fgj["alcaldia_hecho_norm_raw"].fillna(fgj["alcaldia_cat_norm_raw"])
fgj["alcaldia_norm"] = fgj["alcaldia_norm_raw"].map(ALCALDIA_MAP)

# Rango válido y alcaldías válidas
fgj = fgj[fgj["anio_hecho_clean"].between(2016, 2025, inclusive="both")].copy()
fgj = fgj[fgj["alcaldia_norm"].notna()].copy()

print("FGJ limpio:", fgj.shape)
print("Alcaldías:", sorted(fgj["alcaldia_norm"].dropna().unique().tolist()))


C:\Users\salaz\AppData\Local\Temp\ipykernel_16056\2607656476.py:7: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  fgj["fecha_hecho_clean"] = pd.to_datetime(fgj[c_fecha_hecho], errors="coerce", dayfirst=True)


FGJ limpio: (2024266, 31)
Alcaldías: ['Azcapotzalco', 'Benito Juárez', 'Coyoacán', 'Cuajimalpa de Morelos', 'Cuauhtémoc', 'Gustavo A. Madero', 'Iztacalco', 'Iztapalapa', 'La Magdalena Contreras', 'Miguel Hidalgo', 'Milpa Alta', 'Tlalpan', 'Tláhuac', 'Venustiano Carranza', 'Xochimilco', 'Álvaro Obregón']


In [6]:
# -----------------------------
# Categorización de robos patrimoniales
# -----------------------------
def clasificar_robo(delito_norm: str):
    if delito_norm is None or pd.isna(delito_norm):
        return ("Otro delito", "No aplica")

    d = delito_norm

    # Subtipos de robo priorizados
    if "ROBO" in d:
        if "TRANSEUNTE" in d:
            return ("Robo patrimonial", "Robo a transeúnte")
        if "NEGOCIO" in d:
            return ("Robo patrimonial", "Robo a negocio")
        if "CASA HABITACION" in d:
            return ("Robo patrimonial", "Robo a casa habitación")
        if "VEHICULO" in d and "INTERIOR" not in d and "ACCESORIOS" not in d:
            return ("Robo patrimonial", "Robo de vehículo")
        if "ACCESORIOS" in d and "AUTO" in d:
            return ("Robo patrimonial", "Robo de accesorios de auto")
        if "INTERIOR" in d and "VEHICULO" in d:
            return ("Robo patrimonial", "Robo del interior de vehículo")
        # Otros robos que no serán el foco, pero sí robo
        return ("Robo patrimonial", "Otro robo")
    return ("Otro delito", "No aplica")

fgj[["categoria_general", "subtipo_robo"]] = fgj["delito_norm"].apply(
    lambda x: pd.Series(clasificar_robo(x))
)
fgj["is_robo_patrimonial"] = (fgj["categoria_general"] == "Robo patrimonial").astype(int)

fgj["trimestre"] = pd.cut(
    fgj["mes_num"],
    bins=[0, 3, 6, 9, 12],
    labels=["T1", "T2", "T3", "T4"]
)

fgj["anio_hecho_clean"] = fgj["anio_hecho_clean"].astype(int)

fgj[[
    "anio_hecho_clean", "mes_num", "mes_nombre", "alcaldia_norm",
    "categoria_general", "subtipo_robo"
]].head()


,anio_hecho_clean,mes_num,mes_nombre,alcaldia_norm,categoria_general,subtipo_robo
2,2016,1.0,Enero,Iztapalapa,Robo patrimonial,Robo a transeúnte
10,2016,1.0,Enero,Iztapalapa,Otro delito,No aplica
12,2016,1.0,Enero,Miguel Hidalgo,Otro delito,No aplica
15,2016,1.0,Enero,Coyoacán,Robo patrimonial,Robo a casa habitación
16,2016,1.0,Enero,Azcapotzalco,Otro delito,No aplica


In [7]:
# -----------------------------
# Calidad del FGJ limpio para Tableau
# -----------------------------
fgj_qc = pct_null(fgj, [
    "anio_hecho_clean", "mes_num", "mes_nombre", "alcaldia_norm",
    "delito_norm", "categoria_general", "subtipo_robo"
])
fgj_qc


,columna,nulos,porcentaje_nulos
1,mes_num,50519,2.4957
2,mes_nombre,50519,2.4957
0,anio_hecho_clean,0,0.0000
3,alcaldia_norm,0,0.0000
4,delito_norm,0,0.0000
5,categoria_general,0,0.0000
6,subtipo_robo,0,0.0000


In [8]:
# -----------------------------
# FGJ agregado para Tableau
# -----------------------------
fgj_base = (
    fgj.groupby(
        ["anio_hecho_clean", "mes_num", "mes_nombre", "trimestre", "alcaldia_norm", "categoria_general", "subtipo_robo"],
        dropna=False
    )
    .size()
    .reset_index(name="conteo")
)

# Total mensual por alcaldía
total_mes_alcaldia = (
    fgj_base.groupby(["anio_hecho_clean", "mes_num", "mes_nombre", "trimestre", "alcaldia_norm"], dropna=False)["conteo"]
    .sum()
    .reset_index(name="total_delitos_mes_alcaldia")
)

# Total de robos mensuales por alcaldía
robos_mes_alcaldia = (
    fgj_base[fgj_base["categoria_general"] == "Robo patrimonial"]
    .groupby(["anio_hecho_clean", "mes_num", "mes_nombre", "trimestre", "alcaldia_norm"], dropna=False)["conteo"]
    .sum()
    .reset_index(name="total_robos_mes_alcaldia")
)

fgj_tableau = (
    fgj_base
    .merge(total_mes_alcaldia, on=["anio_hecho_clean", "mes_num", "mes_nombre", "trimestre", "alcaldia_norm"], how="left")
    .merge(robos_mes_alcaldia, on=["anio_hecho_clean", "mes_num", "mes_nombre", "trimestre", "alcaldia_norm"], how="left")
)

fgj_tableau["total_robos_mes_alcaldia"] = fgj_tableau["total_robos_mes_alcaldia"].fillna(0).astype(int)
fgj_tableau["share_categoria_sobre_mes_alcaldia"] = np.where(
    fgj_tableau["total_delitos_mes_alcaldia"] > 0,
    fgj_tableau["conteo"] / fgj_tableau["total_delitos_mes_alcaldia"],
    np.nan
)
fgj_tableau["share_robos_mes_alcaldia"] = np.where(
    fgj_tableau["total_delitos_mes_alcaldia"] > 0,
    fgj_tableau["total_robos_mes_alcaldia"] / fgj_tableau["total_delitos_mes_alcaldia"],
    np.nan
)

fgj_tableau = fgj_tableau.sort_values(
    ["anio_hecho_clean", "mes_num", "alcaldia_norm", "categoria_general", "subtipo_robo"]
).reset_index(drop=True)

fgj_tableau.head(10)


C:\Users\salaz\AppData\Local\Temp\ipykernel_16056\3300190326.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fgj.groupby(
C:\Users\salaz\AppData\Local\Temp\ipykernel_16056\3300190326.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fgj_base.groupby(["anio_hecho_clean", "mes_num", "mes_nombre", "trimestre", "alcaldia_norm"], dropna=False)["conteo"]
C:\Users\salaz\AppData\Local\Temp\ipykernel_16056\3300190326.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt

,anio_hecho_clean,mes_num,mes_nombre,trimestre,alcaldia_norm,categoria_general,subtipo_robo,conteo,total_delitos_mes_alcaldia,total_robos_mes_alcaldia,share_categoria_sobre_mes_alcaldia,share_robos_mes_alcaldia
0,2016,1.0,Abril,T1,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN
1,2016,1.0,Abril,T2,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN
2,2016,1.0,Abril,T3,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN
3,2016,1.0,Abril,T4,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN
4,2016,1.0,Abril,NaN,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN
5,2016,1.0,Agosto,T1,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN
6,2016,1.0,Agosto,T2,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN
7,2016,1.0,Agosto,T3,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN
8,2016,1.0,Agosto,T4,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN
9,2016,1.0,Agosto,NaN,Azcapotzalco,Otro delito,No aplica,0,0,0,NaN,NaN


In [9]:
# -----------------------------
# Exportar FGJ para Tableau
# -----------------------------
fgj_out = OUT_DIR / "fgj_tableau_mensual_alcaldia.csv"
fgj_tableau.to_csv(fgj_out, index=False, encoding="utf-8-sig")
print("[OK] Exportado:", fgj_out)
print("Filas:", len(fgj_tableau))


[OK] Exportado: d:\MineriaDatos\DMProject\csv_tableau\fgj_tableau_mensual_alcaldia.csv
Filas: 2163200


## 2. Limpieza de pobreza y construcción del CSV para Tableau

### Diseño del archivo de salida
El archivo `pobreza_tableau_alcaldia_anio.csv` tendrá una fila por:
- alcaldía,
- año.

### Qué contendrá
- indicadores sociales agregados por promedio ponderado,
- métricas de control como número de registros y suma de factores,
- campos listos para usarse en Tableau sin necesidad de recalcular en cada hoja.

Esto permitirá cruzar el contexto social con el comportamiento delictivo de manera clara.


In [10]:
# -----------------------------
# Leer pobreza/MMIP
# -----------------------------
pob, pob_encoding = read_csv_flexible(POB_FILE, low_memory=False)
print(pob.shape)
pob.head(2)


[OK] Leído con encoding: cp1252
(842342, 37)


,numren,sexo,edad,folio,tam_loc,est_dis,upm,factor,ur_rur_2500,municipio,...,E_CENJ,E_rei,E_CASSi,E_CASi,E_nbi,E_cict,E_ett,E_cyt,E_mmip,año
0,1,1,33,1000038011,1,3,10,248,1,1,...,4,5,4,4,5,5,3,5,5,2016
1,2,2,34,1000038011,1,3,10,248,1,1,...,4,5,4,4,5,5,3,5,5,2016


In [11]:
# -----------------------------
# Resolver columnas de pobreza
# -----------------------------
pcols = {normalize_text(c): c for c in pob.columns}

def pcol(*aliases):
    for a in aliases:
        key = normalize_text(a)
        if key in pcols:
            return pcols[key]
    return None

c_anio = pcol("año", "anio")
c_entidad = pcol("entidad")
c_municipio = pcol("municipio")
c_factor = pcol("factor")

social_cols = {
    "mmip": pcol("mmip"),
    "rei": pcol("rei"),
    "nbi": pcol("nbi"),
    "CBDj": pcol("CBDj", "cbdj"),
    "CSj": pcol("CSj", "csj"),
    "CENJ": pcol("CENJ", "cenj"),
    "ett": pcol("ett"),
    "CASi": pcol("CASi", "casi"),
    "CASSi": pcol("CASSi", "cassi"),
    "cyt": pcol("cyt"),
    "ccevj": pcol("ccevj"),
}

social_cols = {k: v for k, v in social_cols.items() if v is not None}
{
    "anio": c_anio,
    "entidad": c_entidad,
    "municipio": c_municipio,
    "factor": c_factor,
    "social_cols": social_cols
}


{'anio': 'año',
 'entidad': 'entidad',
 'municipio': 'municipio',
 'factor': 'factor',
 'social_cols': {'mmip': 'mmip',
  'rei': 'rei',
  'nbi': 'nbi',
  'CBDj': 'CBDj',
  'CSj': 'CSj',
  'CENJ': 'CENJ',
  'ett': 'ett',
  'CASi': 'CASi',
  'CASSi': 'CASSi',
  'cyt': 'cyt',
  'ccevj': 'ccevj'}}

In [12]:
# -----------------------------
# Filtrado a CDMX y mapeo de alcaldías
# -----------------------------
pob[c_anio] = pd.to_numeric(pob[c_anio], errors="coerce")
pob[c_entidad] = pd.to_numeric(pob[c_entidad], errors="coerce")
pob[c_municipio] = pd.to_numeric(pob[c_municipio], errors="coerce")
pob[c_factor] = pd.to_numeric(pob[c_factor], errors="coerce")

# CDMX = entidad 9
pob_cdmx = pob[pob[c_entidad] == 9].copy()
pob_cdmx["alcaldia_norm"] = pob_cdmx[c_municipio].map(MUNICIPIO_CDMX_MAP)

# Mantener solo alcaldías válidas
pob_cdmx = pob_cdmx[pob_cdmx["alcaldia_norm"].notna()].copy()

# Tipos numéricos
for real_col in social_cols.values():
    pob_cdmx[real_col] = pd.to_numeric(pob_cdmx[real_col], errors="coerce")

print("Pobreza CDMX:", pob_cdmx.shape)
print("Años:", sorted(pob_cdmx[c_anio].dropna().astype(int).unique().tolist()))
print("Alcaldías:", sorted(pob_cdmx["alcaldia_norm"].dropna().unique().tolist()))


Pobreza CDMX: (22219, 38)
Años: [2016, 2018, 2020]
Alcaldías: ['Azcapotzalco', 'Benito Juárez', 'Coyoacán', 'Cuajimalpa de Morelos', 'Cuauhtémoc', 'Gustavo A. Madero', 'Iztacalco', 'Iztapalapa', 'La Magdalena Contreras', 'Miguel Hidalgo', 'Milpa Alta', 'Tlalpan', 'Tláhuac', 'Venustiano Carranza', 'Xochimilco', 'Álvaro Obregón']


In [13]:
# -----------------------------
# Calidad de pobreza limpia para Tableau
# -----------------------------
pob_qc = pct_null(pob_cdmx, [c_anio, c_factor, "alcaldia_norm"] + list(social_cols.values()))
pob_qc


,columna,nulos,porcentaje_nulos
0,año,0,0.0
1,factor,0,0.0
2,alcaldia_norm,0,0.0
3,mmip,0,0.0
4,rei,0,0.0
5,nbi,0,0.0
6,CBDj,0,0.0
7,CSj,0,0.0
8,CENJ,0,0.0
9,ett,0,0.0


In [14]:
# -----------------------------
# Agregado ponderado de pobreza por alcaldía-año
# -----------------------------
rows = []
for (anio, alcaldia), sub in pob_cdmx.groupby([c_anio, "alcaldia_norm"], dropna=False):
    row = {
        "anio": int(anio) if pd.notna(anio) else np.nan,
        "alcaldia_norm": alcaldia,
        "n_registros": int(len(sub)),
        "factor_sum": float(sub[c_factor].sum(skipna=True)) if pd.notna(sub[c_factor].sum(skipna=True)) else np.nan
    }
    for logical_name, real_col in social_cols.items():
        row[f"{logical_name}_wmean"] = weighted_mean(sub[real_col], sub[c_factor])
    rows.append(row)

pobreza_tableau = pd.DataFrame(rows).sort_values(["anio", "alcaldia_norm"]).reset_index(drop=True)
pobreza_tableau.head(10)


,anio,alcaldia_norm,n_registros,factor_sum,mmip_wmean,rei_wmean,nbi_wmean,CBDj_wmean,CSj_wmean,CENJ_wmean,ett_wmean,CASi_wmean,CASSi_wmean,cyt_wmean,ccevj_wmean
0,2016,Azcapotzalco,245,393527.0,-0.006374,-0.069260,-0.018415,-0.457162,0.087811,0.005033,1.000432,0.097185,0.305492,0.000820,-0.083150
1,2016,Benito Juárez,242,525216.0,-0.320988,-0.249439,-0.212123,-0.708901,0.020899,0.000000,0.962998,-0.150982,0.201860,-0.386028,-0.325615
2,2016,Coyoacán,277,459883.0,0.010757,-0.081690,0.029540,-0.304103,0.129663,0.000000,0.964209,0.102278,0.381430,-0.000466,0.008096
3,2016,Cuajimalpa de Morelos,169,216080.0,0.097752,0.009551,0.116836,-0.225012,0.225122,0.009180,1.031619,0.129088,0.285255,0.086351,0.200665
4,2016,Cuauhtémoc,589,951496.0,-0.035068,-0.130129,-0.074666,-0.349289,0.021946,0.000307,1.077736,0.042211,0.327553,-0.011411,-0.198367
5,2016,Gustavo A. Madero,672,1016468.0,0.113504,-0.026261,0.051609,-0.326945,0.114592,0.000934,1.040957,0.161293,0.410895,0.150483,-0.000690
6,2016,Iztacalco,286,458083.0,0.072060,-0.099846,0.034347,-0.409135,0.106549,0.006112,1.106485,0.162212,0.414755,0.094591,0.017140
7,2016,Iztapalapa,1080,1667333.0,0.187446,0.009722,0.134874,-0.299521,0.247083,0.001141,1.041733,0.193418,0.538850,0.218855,0.133885
8,2016,La Magdalena Contreras,81,133845.0,0.018614,-0.091742,0.023623,-0.454331,0.131305,0.000000,0.874903,0.051906,0.378468,0.015621,0.043135
9,2016,Miguel Hidalgo,179,322176.0,-0.225123,-0.153587,-0.162799,-0.516127,0.020761,0.000000,0.997224,-0.086063,0.214223,-0.262358,-0.321858


In [15]:
# -----------------------------
# Exportar pobreza para Tableau
# -----------------------------
pob_out = OUT_DIR / "pobreza_tableau_alcaldia_anio.csv"
pobreza_tableau.to_csv(pob_out, index=False, encoding="utf-8-sig")
print("[OK] Exportado:", pob_out)
print("Filas:", len(pobreza_tableau))


[OK] Exportado: d:\MineriaDatos\DMProject\csv_tableau\pobreza_tableau_alcaldia_anio.csv
Filas: 48


## 3. Panel integrado para Tableau

Útil para Tableau porque deja en una sola tabla:
- robos patrimoniales por alcaldía-año,
- total de delitos,
- e indicadores sociales.

Dashboards combinados sin tener que hacer JOIN


In [16]:
# -----------------------------
# Construir panel anual integrado para Tableau
# -----------------------------
fgj_panel = (
    fgj.groupby(["anio_hecho_clean", "alcaldia_norm"], dropna=False)
    .agg(
        total_delitos=("delito_norm", "size"),
        robos_patrimoniales=("is_robo_patrimonial", "sum")
    )
    .reset_index()
    .rename(columns={"anio_hecho_clean": "anio"})
)

panel_tableau = pobreza_tableau.merge(
    fgj_panel,
    on=["anio", "alcaldia_norm"],
    how="left"
)

panel_tableau["total_delitos"] = panel_tableau["total_delitos"].fillna(0).astype(int)
panel_tableau["robos_patrimoniales"] = panel_tableau["robos_patrimoniales"].fillna(0).astype(int)
panel_tableau["share_robos_sobre_total"] = np.where(
    panel_tableau["total_delitos"] > 0,
    panel_tableau["robos_patrimoniales"] / panel_tableau["total_delitos"],
    np.nan
)

panel_out = OUT_DIR / "panel_tableau_alcaldia_anio.csv"
panel_tableau.to_csv(panel_out, index=False, encoding="utf-8-sig")
print("[OK] Exportado:", panel_out)
print("Filas:", len(panel_tableau))
panel_tableau.head(10)


[OK] Exportado: d:\MineriaDatos\DMProject\csv_tableau\panel_tableau_alcaldia_anio.csv
Filas: 48


,anio,alcaldia_norm,n_registros,factor_sum,mmip_wmean,rei_wmean,nbi_wmean,CBDj_wmean,CSj_wmean,CENJ_wmean,ett_wmean,CASi_wmean,CASSi_wmean,cyt_wmean,ccevj_wmean,total_delitos,robos_patrimoniales,share_robos_sobre_total
0,2016,Azcapotzalco,245,393527.0,-0.006374,-0.069260,-0.018415,-0.457162,0.087811,0.005033,1.000432,0.097185,0.305492,0.000820,-0.083150,10093,4877,0.483206
1,2016,Benito Juárez,242,525216.0,-0.320988,-0.249439,-0.212123,-0.708901,0.020899,0.000000,0.962998,-0.150982,0.201860,-0.386028,-0.325615,16550,7917,0.478369
2,2016,Coyoacán,277,459883.0,0.010757,-0.081690,0.029540,-0.304103,0.129663,0.000000,0.964209,0.102278,0.381430,-0.000466,0.008096,13631,6132,0.449857
3,2016,Cuajimalpa de Morelos,169,216080.0,0.097752,0.009551,0.116836,-0.225012,0.225122,0.009180,1.031619,0.129088,0.285255,0.086351,0.200665,2825,987,0.349381
4,2016,Cuauhtémoc,589,951496.0,-0.035068,-0.130129,-0.074666,-0.349289,0.021946,0.000307,1.077736,0.042211,0.327553,-0.011411,-0.198367,31975,12829,0.401220
5,2016,Gustavo A. Madero,672,1016468.0,0.113504,-0.026261,0.051609,-0.326945,0.114592,0.000934,1.040957,0.161293,0.410895,0.150483,-0.000690,19019,8533,0.448657
6,2016,Iztacalco,286,458083.0,0.072060,-0.099846,0.034347,-0.409135,0.106549,0.006112,1.106485,0.162212,0.414755,0.094591,0.017140,8089,3354,0.414637
7,2016,Iztapalapa,1080,1667333.0,0.187446,0.009722,0.134874,-0.299521,0.247083,0.001141,1.041733,0.193418,0.538850,0.218855,0.133885,29337,13053,0.444933
8,2016,La Magdalena Contreras,81,133845.0,0.018614,-0.091742,0.023623,-0.454331,0.131305,0.000000,0.874903,0.051906,0.378468,0.015621,0.043135,3080,975,0.316558
9,2016,Miguel Hidalgo,179,322176.0,-0.225123,-0.153587,-0.162799,-0.516127,0.020761,0.000000,0.997224,-0.086063,0.214223,-0.262358,-0.321858,12631,5767,0.456575


## 4. Control de calidad final

Aquí se revisa que los archivos exportados sean realmente usables:
- sin alcaldías nulas;
- con años válidos;
- con columnas clave completas;
- y con tamaño razonable para Tableau.


In [17]:
# -----------------------------
# Control de calidad final
# -----------------------------
qc_summary = {
    "fgj_encoding": fgj_encoding,
    "pobreza_encoding": pob_encoding,
    "fgj_raw_rows": int(len(fgj)),
    "fgj_tableau_rows": int(len(fgj_tableau)),
    "pobreza_cdmx_rows": int(len(pob_cdmx)),
    "pobreza_tableau_rows": int(len(pobreza_tableau)),
    "panel_tableau_rows": int(len(panel_tableau)),
    "fgj_alcaldias": sorted(fgj_tableau["alcaldia_norm"].dropna().unique().tolist()),
    "pobreza_alcaldias": sorted(pobreza_tableau["alcaldia_norm"].dropna().unique().tolist()),
    "panel_alcaldias": sorted(panel_tableau["alcaldia_norm"].dropna().unique().tolist()),
    "fgj_anios": sorted(pd.to_numeric(fgj_tableau["anio_hecho_clean"], errors="coerce").dropna().astype(int).unique().tolist()),
    "pobreza_anios": sorted(pd.to_numeric(pobreza_tableau["anio"], errors="coerce").dropna().astype(int).unique().tolist()),
    "panel_anios": sorted(pd.to_numeric(panel_tableau["anio"], errors="coerce").dropna().astype(int).unique().tolist()),
    "fgj_nulls": pct_null(fgj_tableau, ["anio_hecho_clean", "mes_num", "alcaldia_norm", "categoria_general", "subtipo_robo", "conteo"]).to_dict(orient="records"),
    "pobreza_nulls": pct_null(pobreza_tableau, ["anio", "alcaldia_norm", "n_registros", "factor_sum"] + [c for c in pobreza_tableau.columns if c.endswith("_wmean")]).to_dict(orient="records"),
    "panel_nulls": pct_null(panel_tableau, ["anio", "alcaldia_norm", "robos_patrimoniales", "total_delitos"]).to_dict(orient="records"),
}

save_json(qc_summary, OUT_DIR / "qc_tableau_outputs.json")
qc_summary


{'fgj_encoding': 'utf-8',
 'pobreza_encoding': 'cp1252',
 'fgj_raw_rows': 2024266,
 'fgj_tableau_rows': 2163200,
 'pobreza_cdmx_rows': 22219,
 'pobreza_tableau_rows': 48,
 'panel_tableau_rows': 48,
 'fgj_alcaldias': ['Azcapotzalco',
  'Benito Juárez',
  'Coyoacán',
  'Cuajimalpa de Morelos',
  'Cuauhtémoc',
  'Gustavo A. Madero',
  'Iztacalco',
  'Iztapalapa',
  'La Magdalena Contreras',
  'Miguel Hidalgo',
  'Milpa Alta',
  'Tlalpan',
  'Tláhuac',
  'Venustiano Carranza',
  'Xochimilco',
  'Álvaro Obregón'],
 'pobreza_alcaldias': ['Azcapotzalco',
  'Benito Juárez',
  'Coyoacán',
  'Cuajimalpa de Morelos',
  'Cuauhtémoc',
  'Gustavo A. Madero',
  'Iztacalco',
  'Iztapalapa',
  'La Magdalena Contreras',
  'Miguel Hidalgo',
  'Milpa Alta',
  'Tlalpan',
  'Tláhuac',
  'Venustiano Carranza',
  'Xochimilco',
  'Álvaro Obregón'],
 'panel_alcaldias': ['Azcapotzalco',
  'Benito Juárez',
  'Coyoacán',
  'Cuajimalpa de Morelos',
  'Cuauhtémoc',
  'Gustavo A. Madero',
  'Iztacalco',
  'Iztapalapa

## 5. Gráficas recomendadas para Tableau

### A. Gráficas con `fgj_tableau_mensual_alcaldia.csv`
1. **Mapa por alcaldía con robos patrimoniales anuales**  
   **Cómo construirla:** usar `alcaldia_norm` como dimensión geográfica o unir con un shapefile de alcaldías; color por suma de `conteo` filtrando `categoria_general = "Robo patrimonial"`.  
   **Por qué es útil:** muestra la distribución territorial del fenómeno principal.

2. **Serie temporal mensual de robos patrimoniales**  
   **Cómo construirla:** eje X con `mes_num` o fecha derivada `anio_hecho_clean + mes_num`; eje Y con suma de `conteo`; filtro por `categoria_general = "Robo patrimonial"`.  
   **Por qué es útil:** permite observar estacionalidad, caídas o repuntes.

3. **Barras apiladas por alcaldía y subtipo de robo**  
   **Cómo construirla:** filas `alcaldia_norm`, columnas suma de `conteo`, color por `subtipo_robo`.  
   **Por qué es útil:** deja ver qué tipo de robo pesa más en cada alcaldía.

4. **Heatmap alcaldía-mes**  
   **Cómo construirla:** filas `alcaldia_norm`, columnas `mes_nombre`, color por suma de `conteo` de robos.  
   **Por qué es útil:** ayuda a identificar concentración temporal y territorial.

5. **Proporción de robos sobre el total delictivo**  
   **Cómo construirla:** líneas o barras con `share_robos_mes_alcaldia`.  
   **Por qué es útil:** permite comparar la importancia relativa de los robos dentro del total delictivo.

### B. Gráficas con `pobreza_tableau_alcaldia_anio.csv`
6. **Mapa por alcaldía de pobreza multidimensional**  
   **Cómo construirla:** color por `mmip_wmean`.  
   **Por qué es útil:** da contexto territorial del perfil social.

7. **Barras comparativas por alcaldía para `mmip_wmean`, `rei_wmean` y `nbi_wmean`**  
   **Cómo construirla:** alcaldía en filas; medida seleccionable o small multiples por indicador.  
   **Por qué es útil:** permite comparar componentes sociales clave.

8. **Serie anual por alcaldía para indicadores sociales**  
   **Cómo construirla:** año en X, indicador en Y, color por alcaldía.  
   **Por qué es útil:** permite ver estabilidad o cambio del contexto social.

### C. Gráficas con `panel_tableau_alcaldia_anio.csv`
9. **Dispersión entre `mmip_wmean` y `robos_patrimoniales`**  
   **Cómo construirla:** eje X `mmip_wmean`, eje Y `robos_patrimoniales`, color por año, etiqueta por alcaldía.  
   **Por qué es útil:** ayuda a visualizar si hay una asociación general o patrones por grupo.

10. **Dispersión entre `rei_wmean` y `robos_patrimoniales`**  
    **Cómo construirla:** igual que la anterior, cambiando la variable del eje X.  
    **Por qué es útil:** permite revisar el papel del rezago educativo.

11. **Barras duales o gráfico combinado por alcaldía**  
    **Cómo construirla:** comparar `robos_patrimoniales` y un indicador social usando ejes sincronizados o dashboard combinado.  
    **Por qué es útil:** comunica bien diferencias territoriales.

12. **Dashboard final del proyecto**  
    **Contenido sugerido:**  
    - mapa de robos,  
    - barras por subtipo de robo,  
    - gráfico de dispersión con pobreza,  
    - tarjeta de métricas con total de robos, alcaldías y años.  
    **Por qué es útil:** resume el proyecto en una sola vista.


In [18]:
# -----------------------------
# Resumen final
# -----------------------------
print("Archivos generados:")
for p in sorted(OUT_DIR.glob("*.csv")):
    print(" -", p.name)
print("\nArchivo de control de calidad:")
print(" - qc_tableau_outputs.json")


Archivos generados:
 - fgj_tableau_mensual_alcaldia.csv
 - panel_tableau_alcaldia_anio.csv
 - pobreza_tableau_alcaldia_anio.csv

Archivo de control de calidad:
 - qc_tableau_outputs.json
